# Úloha č.1:

### Znenie

Naimportujte si potrebné knižnice.

### Riešenie

In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt 
from tqdm import tqdm
from datetime import timedelta
import time

### Vysvetlenie

Naimportované boli balíky knižnice *pytorch*, slúžiace na tvorbu neurónových sietí. Takisto bola naimportovaná knižnica *torchvision* a jej balík *torchvision.transforms*, slúžiaca na prácu s obrázkovými dátami a ich transformácie. Ďalej importované knižnice neslúžia priamo na tvorbu a prácu s neurónovými sieťami.

***

# Úloha č.2:

### Znenie

Implementujte overenie toho, či vaše zariadenia obsahuje CUDA-enabled GPU. Ak áno, použite ho pri práci s vytváranou sieťou.

### Riešenie

In [ ]:
 device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Vysvetlenie

Bol vytvorený objekt *troch.device*, slúžiaci na špecifikovanie zariadenia, na ktorom budú dáta/model siete ukladané. Objekt dostáva ako svoj vstupný parameter názov požadovaného zariadnia, za pomoci ternárneho operátora a funkcie *torch.cuda.is_available()*, ktorá vracia *True*, ak vaše zariadenie obsahuje CUDA-enabled GPU.

***

# Úloha č.3:

### Znenie

Získajte dataset CIFAR10 ponúkaný balíkom torchvision a implementujte DataLoader, ktorý obaľuje jeho dáta.

Dokumentácia k datasetu: https://pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html#torchvision.datasets.CIFAR10

### Riešenie

In [ ]:
batch_size = 100

# dataset with training data
train_dataset = torchvision.datasets.CIFAR10(
    root='./Data',
    download=True,
    train=True,
    transform=transforms.ToTensor()
)

# dataset with testing data
test_dataset = torchvision.datasets.CIFAR10(
    root='./Data',
    download=True,
    train=False,
    transform=transforms.ToTensor()
)

# dataloader for training data
train_loader = torch.utils.data.DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True
)

# dataloader for testing data
test_loader = torch.utils.data.DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    shuffle=False
)

### Vysvetlenie

Vytvorené boli dva *Dataset* objekty datasetu CIFAR10, jeden obsahujúci trénovacie a druhý testovacie dáta, podľa parametru konštruktora. Oba datasety vykonávajú transformáciu *ToTensor()*, ktorá obrázkové dáta transformuje na Pytorch tensory. Pre ne boli následne vytvorené dva *DataLoader* objekty. 

Dáta obsiahnuté v takomto datasete majú tvar (3, 32, 32), keďže ide o RGB obrázky o veľkosti 32\*32.

***

# Úloha č.4: 

### Znenie

Na základe vzorovej úlohy implementujte model CNN, ktorý obsahuje aspoň jeden konvlučný blok. Experimentujte s rôznymi kombináciami vrstiev.

**Nezabudnite správne určiť hyperparametre siete (na veľkosti vstupných obrázkov a počtu tried)**

### Riešenie

In [ ]:
num_classes = 10

# CNN specific hyperparameters
in_channels = 3
kernel_size = 5     
stride = 1
padding = 2

class MyCNN(nn.Module):
    def __init__(self, in_channels, kernel_size, stride, padding, num_classes):     
        super(MyCNN, self).__init__()

        self.conv1 = nn.Sequential(         
            nn.Conv2d(
                in_channels= in_channels,          
                out_channels= 16,
                kernel_size= kernel_size,         
                stride= stride,
                padding= padding,
            ),                              
            nn.ReLU(),                      
            nn.MaxPool2d(kernel_size=2)
        )

        self.conv2 = nn.Sequential(         
            nn.Conv2d(
                in_channels=16,
                out_channels=32,            
                kernel_size= kernel_size,              
                stride= stride,                   
                padding= padding,                  
            ),  
            nn.ReLU(),                      
            nn.MaxPool2d(kernel_size=2)             
        )
        
        self.conv3 = nn.Sequential(         
            nn.Conv2d(
                in_channels=32,
                out_channels=64,            
                kernel_size= kernel_size,              
                stride= stride,                   
                padding= padding,                  
            ),  
            nn.ReLU(),                      
            nn.MaxPool2d(kernel_size=2)             
        )

        self.fc_output = nn.Linear(1024, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)

        x = x.view(x.size(0), -1)       
        output = self.fc_output(x)
        return output
    
model = CNN(in_channels, kernel_size, stride, padding, num_classes).to(device) # initialize a new network with our hyperparameters (on our specified devide, to utilize GPU if available)

print(model)

### Vysvetlenie

Vytvorená CNN sieť obsahuje 3 konvolučné bloky, pričom každý z nich obsahuje konvolučnú vrstvu, aktivačnú funkciu ReLU a MaxPooling vrstvu. Bloky boli implementované pomocou triedy *nn.Sequential*. Táto trieda zaobaľuje viacero vrstiev do jedného objektu a pri dopredom prechode sekvenčene aplikuje každú zo zaobalených vrstiev na dáta. Každý z blokov obsahuje konvolučnú vrstvu s parametrami *kernel_size*, *stride* a *padding* udávaných podľa hyperparametrov siete. 

Počas dopredného prechodu dáta teda postupne prechádzajú cez všetky konvolučné bloky, pričom sú spracováváné konvolúciou, až na konci je pomocou lineárnej vrstvy vykonaná samotná klasfikácia. Všimnite si, že pretvarovanie dát sa vykonáva až po prechode konvolučnými blokmi, a nie na začiatku popredného prechodu, ako to bolo pri predchádzajúcom prípklade feed-forward siete. Je to preto, že konvolučné vrstvy pracujú s viacrozmernými dátami, takže vyrovnávanie dát je potrebné vykonať až pred prechodom lineárnou vrstvou.

***

# Úloha č.5:

### Znenie

Natrénujte vašu sieť pomocou už známej trénovacej slučky. Môžte experimentovať napr. s výberom rôzych optimilzátorov alebo počtom epoch.

### Riešenie

In [ ]:
model = MyCNN(in_channels, kernel_size, stride, padding, num_classes).to(device)

num_epochs = 30
learning_rate = 0.002

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

start_time = time.time()
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1} training progress:")

    for i, (images, labels) in enumerate(tqdm(train_loader)): 
        labels = labels.to(device)
        images = images.to(device)
        
        outputs = model(images)

        optimizer.zero_grad()
        
        loss = criterion(outputs, labels)
        loss.backward()
        
        optimizer.step()
    
    if epoch + 1 == num_epochs: 
        print (f'\nTraining took {str(timedelta(seconds=time.time() - start_time))}, loss after all epochs: {loss.item():.5f}\n')    
    else:
        print (f'Loss at end of epoch {epoch+1}: {loss.item():.5f}\n')

### Vysvetlenie

Vytvorená trénovacia slučka má rovnakú štruktúru, ako už bolo viackrát prezentované na minulých cvičeniach, preto sa ňou nebudeme do hĺbky zaoberať. Jediné, čo stojí za povšimnutie je fakt, že dáta nie sú pred vsetupom do siete nijak pretvarovávnané (tak ako v príklade feed-forward sietí na dávky jednorozmerných tensorov). Vyplýva to z toho, že v tomto prípade ide o sieť obsahujúcu konvolučné vrstvy, a tie sú schopné pracovať s dávkami viacrozmerných tensorov.

***

# Úloha č.6:

### Znenie

Keď máte sieť natrénovanú, otestujte ju. Testovanie vykonávajte pomocou už známeho postupu (pomer správnych predikcií), prípadne môžte zvoliť inú, vlastnú metriku.

### Riešenie

In [ ]:
with torch.no_grad():
    correct_count = 0
    total_count = 0

    for images, labels in tqdm(test_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        _, predicted_labels = torch.max(outputs.data, 1)
        
        total_count += labels.size(0)
        correct_count += (predicted_labels == labels).sum()
        
accuracy = 100. * correct_count / total_count
print (f'Measured accuracy: {accuracy:0.3f}% \n')

### Vysvetlenie

Trénovacia slučka má už známu štruktúru, pričom iteruje cez testovacie dáta a na základe vykonaných predikcií vypočítava výslednú presnosť natrénovanej siete.